# PPO Poker Agent — Behavioral Analysis

This notebook analyzes the trained PPO agent's decision-making across 507 hand/scenario combinations against GGPoker 200NL heads-up preflop GTO charts. Key questions:

1. Where does the agent match GTO, and where does it diverge?
2. What hand categories does it handle best/worst?
3. How confident is the agent in its decisions (probability mass on chosen action)?
4. Counterfactual: if the agent had played GTO on wrong hands, how would accuracy change by scenario?

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch

# Load pre-computed results from best model
RESULTS_PATH = os.path.join("..", "models", "BEST_GTO__optA_46pct_accuracy", "gto_results.csv")
df = pd.read_csv(RESULTS_PATH)
print(f"Loaded {len(df)} hand/scenario evaluations")
df.head()

## 1. Overall Accuracy by Scenario

In [ ]:
SCENARIO_LABELS = {
    "SB_RFI":     "SB Opening",
    "BB_vs_RFI":  "BB vs SB Open",
    "SB_vs_3bet": "SB vs BB 3-Bet",
}

overall = df["correct"].mean()
print(f"Overall GTO accuracy: {overall:.1%}")
print()

scenario_acc = []
for sc, label in SCENARIO_LABELS.items():
    sub = df[df["scenario"] == sc]
    acc = sub["correct"].mean()
    n   = len(sub)
    scenario_acc.append((label, acc, n))
    print(f"  {label:<22} {acc:.1%}  ({n} hands)")

labels, accs, _ = zip(*scenario_acc)
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(labels, [a*100 for a in accs], color=["steelblue", "tomato", "seagreen"])
ax.axhline(33.3, color="gray", linestyle="--", label="Random baseline (33%)")
ax.axhline(overall*100, color="black", linestyle="-.", label=f"Overall ({overall:.1%})")
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f"{acc:.1%}", ha="center", va="bottom", fontweight="bold")
ax.set_ylabel("GTO Accuracy (%)")
ax.set_title("Agent GTO Accuracy by Scenario")
ax.set_ylim(0, 105)
ax.legend()
plt.tight_layout()
plt.show()

## 2. Accuracy by Hand Category

Breaking down accuracy into pocket pairs, suited hands, and offsuit hands across scenarios.

In [ ]:
def hand_category(row):
    if row["rank1"] == row["rank2"]:
        return "Pocket Pair"
    if row["suited"]:
        return "Suited"
    return "Offsuit"

df["category"] = df.apply(hand_category, axis=1)

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle("Accuracy by Hand Category and Scenario", fontsize=13)

cats = ["Pocket Pair", "Suited", "Offsuit"]
cat_colors = {"Pocket Pair": "gold", "Suited": "steelblue", "Offsuit": "tomato"}

for ax, (sc, label) in zip(axes, SCENARIO_LABELS.items()):
    sub = df[df["scenario"] == sc]
    cat_accs = [sub[sub["category"] == c]["correct"].mean() for c in cats]
    cat_ns   = [len(sub[sub["category"] == c]) for c in cats]
    bars = ax.bar(cats, [a*100 for a in cat_accs],
                  color=[cat_colors[c] for c in cats])
    ax.axhline(33.3, color="gray", linestyle="--", alpha=0.6)
    for bar, acc, n in zip(bars, cat_accs, cat_ns):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f"{acc:.1%}\n(n={n})", ha="center", va="bottom", fontsize=9)
    ax.set_title(label)
    ax.set_ylabel("GTO Accuracy (%)")
    ax.set_ylim(0, 115)

plt.tight_layout()
plt.show()

## 3. Agent Confidence Analysis

How certain is the agent on correct vs incorrect predictions? High confidence on wrong hands indicates systematic bias.

In [ ]:
# Confidence = probability mass on the chosen action
action_prob_map = {"fold": "fold_prob", "call": "call_prob", "raise": "raise_prob"}

def agent_confidence(row):
    return row[action_prob_map[row["agent_action"]]]

df["confidence"] = df.apply(agent_confidence, axis=1)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Distribution: correct vs incorrect
ax = axes[0]
correct_conf   = df[df["correct"]]["confidence"]
incorrect_conf = df[~df["correct"]]["confidence"]
ax.hist(correct_conf,   bins=20, alpha=0.7, color="seagreen", label=f"Correct (n={len(correct_conf)})")
ax.hist(incorrect_conf, bins=20, alpha=0.7, color="tomato",   label=f"Incorrect (n={len(incorrect_conf)})")
ax.axvline(correct_conf.mean(),   color="seagreen", linestyle="--", label=f"Correct mean: {correct_conf.mean():.2f}")
ax.axvline(incorrect_conf.mean(), color="tomato",   linestyle="--", label=f"Incorrect mean: {incorrect_conf.mean():.2f}")
ax.set_xlabel("Agent Confidence (prob mass on chosen action)")
ax.set_ylabel("Count")
ax.set_title("Agent Confidence: Correct vs Incorrect")
ax.legend()

# Confidence by scenario
ax = axes[1]
scenario_names = list(SCENARIO_LABELS.values())
scenario_keys  = list(SCENARIO_LABELS.keys())
for i, (sc, label) in enumerate(SCENARIO_LABELS.items()):
    sub_c = df[(df["scenario"] == sc) & df["correct"]]["confidence"]
    sub_w = df[(df["scenario"] == sc) & ~df["correct"]]["confidence"]
    x_base = i * 2.5
    ax.bar(x_base,     sub_c.mean(), color="seagreen", alpha=0.8, width=0.9, label="Correct" if i==0 else "")
    ax.bar(x_base+1.0, sub_w.mean(), color="tomato",   alpha=0.8, width=0.9, label="Incorrect" if i==0 else "")
    ax.text(x_base,     sub_c.mean()+0.01, f"{sub_c.mean():.2f}", ha="center", fontsize=9)
    ax.text(x_base+1.0, sub_w.mean()+0.01, f"{sub_w.mean():.2f}", ha="center", fontsize=9)

ax.set_xticks([0.5, 3.0, 5.5])
ax.set_xticklabels(scenario_names, rotation=10)
ax.set_ylabel("Mean Confidence")
ax.set_title("Mean Agent Confidence by Scenario")
ax.set_ylim(0, 1.1)
ax.legend()

plt.tight_layout()
plt.show()

print(f"\nOverall mean confidence (correct):   {correct_conf.mean():.3f}")
print(f"Overall mean confidence (incorrect): {incorrect_conf.mean():.3f}")

## 4. Action Bias Analysis — Over-fold / Over-raise Tendencies

Compare agent action frequencies vs GTO target frequencies per scenario.

In [ ]:
actions = ["fold", "call", "raise"]
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle("Action Frequency: Agent vs GTO per Scenario", fontsize=13)

for ax, (sc, label) in zip(axes, SCENARIO_LABELS.items()):
    sub = df[df["scenario"] == sc]
    gto_freq   = [np.mean(sub["gto_action"] == a) for a in actions]
    agent_freq = [np.mean(sub["agent_action"] == a) for a in actions]
    x = np.arange(len(actions))
    w = 0.35
    b1 = ax.bar(x - w/2, [f*100 for f in gto_freq],   w, label="GTO",   color="seagreen", alpha=0.85)
    b2 = ax.bar(x + w/2, [f*100 for f in agent_freq], w, label="Agent", color="steelblue", alpha=0.85)
    for bar, val in zip(b1, gto_freq):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5, f"{val:.1%}",
                ha="center", fontsize=8, color="seagreen")
    for bar, val in zip(b2, agent_freq):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5, f"{val:.1%}",
                ha="center", fontsize=8, color="steelblue")
    ax.set_xticks(x)
    ax.set_xticklabels([a.capitalize() for a in actions])
    ax.set_title(label)
    ax.set_ylabel("% of hands")
    ax.set_ylim(0, 105)
    ax.legend()

plt.tight_layout()
plt.show()

# Print deltas
print("\nAction frequency delta (Agent − GTO):")
for sc, label in SCENARIO_LABELS.items():
    sub = df[df["scenario"] == sc]
    print(f"  {label}:")
    for a in actions:
        delta = np.mean(sub["agent_action"] == a) - np.mean(sub["gto_action"] == a)
        direction = "↑" if delta > 0.01 else ("↓" if delta < -0.01 else "≈")
        print(f"    {a:<6}: {delta:+.1%} {direction}")

## 5. Top Misclassified Hands — Error Analysis

In [ ]:
wrong = df[~df["correct"]].copy()
wrong["error_type"] = wrong["gto_action"] + "→" + wrong["agent_action"]

print("Most common error types:")
print(wrong.groupby(["scenario", "error_type"]).size().reset_index(name="count")
      .sort_values("count", ascending=False).head(15).to_string(index=False))

print("\nTop misclassified hands (any scenario):")
top = (wrong.groupby(["hand", "scenario", "gto_action", "agent_action"])
            .agg(count=("correct", "count"), avg_confidence=("confidence", "mean"))
            .reset_index()
            .sort_values("count", ascending=False)
            .head(20))
print(top.to_string(index=False))

## 6. Counterfactual Analysis — Per-Scenario Accuracy if Agent Were Corrected

Simulates: "What would overall accuracy be if the agent's biggest mistake category were fixed?"

In [ ]:
for sc, label in SCENARIO_LABELS.items():
    sub = df[df["scenario"] == sc].copy()
    base_acc = sub["correct"].mean()
    print(f"\n{label} — base accuracy: {base_acc:.1%}")

    # For each error type, simulate "what if this error were corrected"
    wrong_sub = sub[~sub["correct"]]
    error_types = wrong_sub.groupby("error_type").size().sort_values(ascending=False)

    for err_type, n_err in error_types.items():
        # hypothetical accuracy if this error were fixed
        hypothetical = (sub["correct"].sum() + n_err) / len(sub)
        gain = hypothetical - base_acc
        print(f"  Fix '{err_type}' ({n_err} hands) → {hypothetical:.1%} (+{gain:.1%})")

## 7. Premium Hand Verification

Key sanity check: does the agent correctly play the most important hands?

In [ ]:
premium_hands = ["AA", "KK", "QQ", "JJ", "TT", "AKs", "AKo", "AQs", "72o", "32o"]

print("Premium / landmark hand verification:")
print(f"{'Hand':<6} {'Scenario':<22} {'GTO':<6} {'Agent':<6} {'Match':<6} {'Fold%':>7} {'Call%':>7} {'Raise%':>8}")
print("-" * 75)

for hand in premium_hands:
    rows = df[df["hand"] == hand]
    for _, r in rows.iterrows():
        match = "✓" if r["correct"] else "✗"
        print(f"{r['hand']:<6} {SCENARIO_LABELS[r['scenario']]:<22} {r['gto_action']:<6} {r['agent_action']:<6} "
              f"{match:<6} {r['fold_prob']*100:>7.1f} {r['call_prob']*100:>7.1f} {r['raise_prob']*100:>8.1f}")

## 8. Rank-by-Rank Accuracy Heatmap

Which card ranks contribute most to correct vs incorrect classifications?

In [ ]:
RANK_NAMES = {14: "A", 13: "K", 12: "Q", 11: "J", 10: "T",
              9: "9", 8: "8", 7: "7", 6: "6", 5: "5", 4: "4", 3: "3", 2: "2"}

ranks = list(range(14, 1, -1))
rank_labels = [RANK_NAMES[r] for r in ranks]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("GTO Accuracy Heatmap by Hand (Agent Prediction)", fontsize=13)
cmap = mcolors.LinearSegmentedColormap.from_list("rg", ["tomato", "gold", "seagreen"])

for ax, (sc, label) in zip(axes, SCENARIO_LABELS.items()):
    sub = df[df["scenario"] == sc]
    grid = np.full((13, 13), np.nan)
    for _, row in sub.iterrows():
        r1i = ranks.index(int(row["rank1"]))
        r2i = ranks.index(int(row["rank2"]))
        grid[r1i][r2i] = float(row["correct"])

    im = ax.imshow(grid, cmap=cmap, vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(13)); ax.set_xticklabels(rank_labels, fontsize=8)
    ax.set_yticks(range(13)); ax.set_yticklabels(rank_labels, fontsize=8)
    ax.set_xlabel("Low Card")
    ax.set_ylabel("High Card")
    ax.set_title(label)
    plt.colorbar(im, ax=ax, label="Correct")

plt.tight_layout()
plt.show()